# micrograd, from scratch

This is the *Practice* step of `unit_01_micrograd.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the real micrograd repo.
- Stuck on an *idea* for 20 min → read one tier of `HINTS.md`, or ask.
- Stuck on *Python syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Later milestones add methods to `Value` with `Value.name = name`. That's just so you
can build the class up one cell at a time instead of re-running one giant cell.

In [ ]:
import math
import random
from test_micrograd import grade

## Milestone 1 & 2 — a number that remembers, and one hop of gradient

Each op does two jobs: compute the forward number, and attach a closure that knows the
local derivative. Write both in the same method.

The container is given. The ideas are the two methods below it.

In [ ]:
class Value:
    """A scalar that remembers the operation that produced it.

    Fields:
      data       float, the actual number
      grad       float, d(final output) / d(self). starts at zero.
      _prev      set of Values that were the inputs to the op producing self
      _op        str, debug label for that op
      _backward  a closure that takes self.grad and pushes gradient ONE HOP
                 back into each element of _prev. does nothing for a leaf.
    """

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def dump(self, indent=0):
        """Plumbing: print the expression graph as text. For debugging."""
        pad = '  ' * indent
        label = self._op or 'leaf'
        print(f"{pad}{label:>6} data={self.data:<12.6g} grad={self.grad:<12.6g}")
        for child in self._prev:
            child.dump(indent + 1)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        raise NotImplementedError

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        raise NotImplementedError

In [ ]:
grade(Value, upto=2)

## Milestone 3 & 4 — the whole graph, in the right order

Two questions to answer before you write a line:
1. In what order must the nodes be visited, and why that order?
2. What is `self.grad`, before any of this starts?

Milestone 4 has no new stub. It's a one-character change somewhere above, and the grader will tell you when you need it.

In [ ]:
def backward(self):
    """Run backprop from self through the entire graph behind it."""
    raise NotImplementedError

Value.backward = backward

In [ ]:
grade(Value, upto=4)

## Milestone 5 — tanh

You get to treat this as a single atomic op with one local derivative, even though
it is really exp/div/sub underneath.

In [ ]:
def tanh(self):
    raise NotImplementedError

Value.tanh = tanh

In [ ]:
grade(Value, upto=5)

## Milestone 6 (stretch) — granularity

`exp` and `pow` need real `_backward` closures. Everything after them does **not**:
each is a one-liner built out of ops you already have. If you find yourself writing a
closure for `__sub__`, stop and think.

The four `__r*__` methods fire when the Value is on the *right* of the operator,
e.g. `2.0 * v` or `2.0 - v`. Pure Python trivia, ask if annoying.

In [ ]:
def exp(self):
    raise NotImplementedError

def __pow__(self, other):
    assert isinstance(other, (int, float)), "only scalar exponents"
    raise NotImplementedError

def __neg__(self):
    raise NotImplementedError

def __sub__(self, other):
    raise NotImplementedError

def __truediv__(self, other):
    raise NotImplementedError

def __radd__(self, other):
    raise NotImplementedError

def __rmul__(self, other):
    raise NotImplementedError

def __rsub__(self, other):
    raise NotImplementedError

def __rtruediv__(self, other):
    raise NotImplementedError

for _f in (exp, __pow__, __neg__, __sub__, __truediv__,
           __radd__, __rmul__, __rsub__, __rtruediv__):
    setattr(Value, _f.__name__, _f)

In [ ]:
grade(Value, upto=6)

## Milestone 7 (stretch) — it learns

Nothing below here knows anything about calculus. That is the point: once `Value`
works, a neural net is just arithmetic on top of it.

In [ ]:
class Neuron:
    def __init__(self, nin):
        """nin weights and one bias. init weights uniform in [-1, 1]."""
        raise NotImplementedError

    def __call__(self, x):
        """x is a list of nin floats or Values. Return one Value: the
        squashed weighted sum."""
        raise NotImplementedError

    def parameters(self):
        """Flat list of every Value the optimizer is allowed to nudge."""
        raise NotImplementedError


class Layer:
    def __init__(self, nin, nout):
        raise NotImplementedError

    def __call__(self, x):
        """Return a list of Values -- or a bare Value if nout == 1."""
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError


class MLP:
    def __init__(self, nin, nouts):
        """nouts is a list of layer widths, e.g. MLP(3, [4, 4, 1])."""
        raise NotImplementedError

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError

In [ ]:
grade(Value, Neuron, Layer, MLP)

## Your own training loop

The grader ran a training loop for you in milestone 7. Now write one yourself, from
memory, on the same toy data. Order of operations matters — see `HINTS.md` milestone 7
if the loss does something weird.

In [ ]:
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
model = MLP(3, [4, 4, 1])

for step in range(50):
    ...

## Scratch

Space to poke at things. `v.dump()` prints the graph behind any Value.